# Mini-project: Advanced Statistical Analysis of Apple Inc. (AAPL) Stock Data

Ce notebook présente une analyse statistique avancée du cours de l’action Apple à partir d’un fichier CSV historique.  
L’objectif est d’utiliser **Pandas**, **NumPy**, **Matplotlib** et **SciPy** pour explorer les données, visualiser les tendances, calculer des statistiques, tester des hypothèses et appliquer quelques techniques avancées.

## Objectifs
- Charger et nettoyer les données
- Étudier les propriétés de la série temporelle
- Visualiser les prix et le volume échangé
- Calculer des statistiques descriptives
- Effectuer des tests d’hypothèses
- Explorer des techniques avancées avec NumPy et SciPy

## 1) Import des bibliothèques

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from scipy.signal import savgol_filter

plt.style.use("ggplot")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

## 2) Chargement des données

Placez le fichier CSV `AAPL.csv` dans le même dossier que ce notebook, puis exécutez la cellule suivante.

In [ ]:
file_path = "AAPL.csv"
df = pd.read_csv(file_path)

df.head()

### Vérification rapide de la structure

In [ ]:
print("Shape:", df.shape)
display(df.head())
df.info()

## 3) Préparation et exploration initiale

La colonne `Date` est convertie au format datetime, puis utilisée comme index pour faciliter l’analyse temporelle.

In [ ]:
# Conversion de la colonne Date
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

# Suppression des lignes où la date est invalide
df = df.dropna(subset=["Date"]).copy()

# Tri chronologique
df = df.sort_values("Date").set_index("Date")

df.head()

### Valeurs manquantes et types de données

In [ ]:
print("Valeurs manquantes par colonne:")
display(df.isnull().sum())

print("\nTypes de données:")
display(df.dtypes)

### Résumé temporel

In [ ]:
print("Date minimale :", df.index.min())
print("Date maximale :", df.index.max())
print("Nombre d'années couvertes :", df.index.year.nunique())
print("Nombre total d'observations :", len(df))

### Aperçu statistique général

In [ ]:
display(df.describe(include="all"))

## 4) Visualisation des données

Nous allons tracer l’évolution du prix de clôture, du volume échangé, ainsi que des prix haut/bas.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(df.index, df["Close"], label="Close")
ax.set_title("Apple Inc. - Prix de clôture")
ax.set_xlabel("Date")
ax.set_ylabel("Prix")
ax.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(df.index, df["Volume"], label="Volume")
ax.set_title("Apple Inc. - Volume échangé")
ax.set_xlabel("Date")
ax.set_ylabel("Volume")
ax.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(df.index, df["High"], label="High")
ax.plot(df.index, df["Low"], label="Low")
ax.set_title("Apple Inc. - Prix haut et bas")
ax.set_xlabel("Date")
ax.set_ylabel("Prix")
ax.legend()
plt.show()

### Chandeliers (Candlestick chart)

Si `mplfinance` n’est pas installé, exécutez la cellule d’installation ci-dessous.

In [ ]:
# Décommentez si nécessaire :
# !pip install mplfinance

In [ ]:
try:
    import mplfinance as mpf

    # mplfinance attend des colonnes Open, High, Low, Close et un index datetime
    data_candle = df[["Open", "High", "Low", "Close", "Volume"]].copy().tail(250)

    mpf.plot(
        data_candle,
        type="candle",
        volume=True,
        style="yahoo",
        title="Apple Inc. - Candlestick Chart (250 dernières observations)",
        figsize=(14, 8)
    )
except Exception as e:
    print("Impossible d'afficher le candlestick chart :", e)

## 5) Analyse statistique

Nous calculons ici des statistiques descriptives, les rendements journaliers et des moyennes mobiles.

In [ ]:
# Sélection des colonnes numériques
numeric_cols = ["Open", "High", "Low", "Close", "Adj Close", "Volume"]
available_cols = [c for c in numeric_cols if c in df.columns]

stats_summary = pd.DataFrame({
    "mean": df[available_cols].mean(numeric_only=True),
    "median": df[available_cols].median(numeric_only=True),
    "std": df[available_cols].std(numeric_only=True),
    "min": df[available_cols].min(numeric_only=True),
    "max": df[available_cols].max(numeric_only=True),
})

display(stats_summary)

In [ ]:
# Rendements journaliers
df["Daily_Return"] = df["Close"].pct_change()

# Moyennes mobiles
df["MA20"] = df["Close"].rolling(window=20).mean()
df["MA50"] = df["Close"].rolling(window=50).mean()
df["MA200"] = df["Close"].rolling(window=200).mean()

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(df.index, df["Close"], label="Close", alpha=0.7)
ax.plot(df.index, df["MA20"], label="MA20")
ax.plot(df.index, df["MA50"], label="MA50")
ax.plot(df.index, df["MA200"], label="MA200")
ax.set_title("Prix de clôture et moyennes mobiles")
ax.set_xlabel("Date")
ax.set_ylabel("Prix")
ax.legend()
plt.show()

### Distribution des rendements journaliers

In [ ]:
returns = df["Daily_Return"].dropna()

fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(returns, bins=60)
ax.set_title("Distribution des rendements journaliers")
ax.set_xlabel("Rendement")
ax.set_ylabel("Fréquence")
plt.show()

print("Skewness :", stats.skew(returns))
print("Kurtosis :", stats.kurtosis(returns))

## 6) Tests d’hypothèse

### 6.1 Comparaison des prix de clôture entre deux années

Exemple : comparaison entre 2022 et 2023.  
L’hypothèse nulle \(H_0\) : les moyennes des prix de clôture sont identiques.  
L’hypothèse alternative \(H_1\) : les moyennes sont différentes.

In [ ]:
year1 = 2022
year2 = 2023

close_y1 = df[df.index.year == year1]["Close"].dropna()
close_y2 = df[df.index.year == year2]["Close"].dropna()

print(f"Nombre d'observations {year1} :", len(close_y1))
print(f"Nombre d'observations {year2} :", len(close_y2))

t_stat, p_value = stats.ttest_ind(close_y1, close_y2, equal_var=False, nan_policy="omit")

print("T-statistic :", t_stat)
print("P-value :", p_value)

alpha = 0.05
if p_value < alpha:
    print("Conclusion : on rejette H0. Les moyennes sont significativement différentes.")
else:
    print("Conclusion : on ne rejette pas H0. Pas de différence significative détectée.")

### 6.2 Normalité des rendements journaliers

On utilise un échantillon pour le test de Shapiro-Wilk, qui devient très coûteux sur de grandes séries.

In [ ]:
sample_size = min(5000, len(returns))
sample_returns = returns.sample(sample_size, random_state=42)

shapiro_stat, shapiro_p = stats.shapiro(sample_returns)

print("Shapiro-Wilk statistic :", shapiro_stat)
print("Shapiro-Wilk p-value    :", shapiro_p)

if shapiro_p < 0.05:
    print("Conclusion : les rendements ne suivent pas une loi normale.")
else:
    print("Conclusion : la normalité n'est pas rejetée.")

In [ ]:
# Test complémentaire de normalité : D'Agostino-Pearson
if len(returns) >= 20:
    dagostino_stat, dagostino_p = stats.normaltest(returns.dropna())
    print("D'Agostino statistic :", dagostino_stat)
    print("D'Agostino p-value   :", dagostino_p)
else:
    print("Échantillon trop petit pour le test de normalité de D'Agostino-Pearson.")

## 7) Techniques statistiques avancées

### 7.1 Moyenne mobile avec `np.convolve`

In [ ]:
window = 20
kernel = np.ones(window) / window
close_values = df["Close"].dropna().values

ma_convolve = np.convolve(close_values, kernel, mode="valid")

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(df.index, df["Close"], label="Close", alpha=0.6)
ax.plot(df.index[window - 1:], ma_convolve, label=f"MA{window} via convolve")
ax.set_title("Moyenne mobile calculée avec NumPy")
ax.set_xlabel("Date")
ax.set_ylabel("Prix")
ax.legend()
plt.show()

### 7.2 Corrélations entre variables financières

In [ ]:
corr_cols = [c for c in ["Open", "High", "Low", "Close", "Adj Close", "Volume"] if c in df.columns]
corr_matrix = df[corr_cols].corr(numeric_only=True)

display(corr_matrix)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr_matrix.values, aspect="auto")
ax.set_xticks(range(len(corr_cols)))
ax.set_yticks(range(len(corr_cols)))
ax.set_xticklabels(corr_cols, rotation=45, ha="right")
ax.set_yticklabels(corr_cols)
ax.set_title("Matrice de corrélation")

fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
# Corrélation entre prix de clôture et volume
if "Volume" in df.columns:
    corr_close_volume = np.corrcoef(df["Close"].dropna().align(df["Volume"].dropna(), join="inner")[0],
                                    df["Close"].dropna().align(df["Volume"].dropna(), join="inner")[1])[0, 1]
    print("Corrélation Close / Volume :", corr_close_volume)
else:
    print("La colonne Volume est absente.")

### 7.3 Corrélation entre moyenne mobile et volume

In [ ]:
if "MA50" in df.columns and "Volume" in df.columns:
    temp = df[["MA50", "Volume"]].dropna()
    corr_ma50_volume = temp["MA50"].corr(temp["Volume"])
    print("Corrélation MA50 / Volume :", corr_ma50_volume)
else:
    print("Impossible de calculer la corrélation MA50 / Volume.")

### 7.4 Lissage de la série par filtre de Savitzky-Golay

In [ ]:
# La longueur de fenêtre doit être impaire et inférieure à la taille de l'échantillon
close_series = df["Close"].dropna().values

if len(close_series) >= 51:
    smoothed = savgol_filter(close_series, window_length=51, polyorder=3)

    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(df.index[:len(close_series)], close_series, label="Close", alpha=0.5)
    ax.plot(df.index[:len(close_series)], smoothed, label="Smoothed")
    ax.set_title("Lissage Savitzky-Golay")
    ax.set_xlabel("Date")
    ax.set_ylabel("Prix")
    ax.legend()
    plt.show()
else:
    print("Pas assez d'observations pour appliquer un filtre Savitzky-Golay de longueur 51.")

## 8) Analyse complémentaire : comparaison par périodes

On peut aussi étudier les prix de clôture par grandes périodes pour mieux comprendre l'évolution à long terme.

In [ ]:
df["Year"] = df.index.year
annual_close = df.groupby("Year")["Close"].mean()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(annual_close.index, annual_close.values)
ax.set_title("Prix moyen annuel de clôture")
ax.set_xlabel("Année")
ax.set_ylabel("Close moyen")
plt.show()

display(annual_close.tail(10))

## 9) Synthèse des résultats

### Points principaux
- Apple présente une tendance haussière de long terme.
- Les moyennes mobiles aident à lisser le bruit du marché.
- Les rendements journaliers sont souvent asymétriques et non parfaitement normaux.
- Les prix Open, High, Low et Close sont fortement corrélés.
- Le volume est utile pour contextualiser l’activité du marché, mais il est souvent moins corrélé aux prix que les variables de prix elles-mêmes.

### Interprétation financière
L’action AAPL a connu une forte croissance sur la période étudiée.  
Les outils statistiques permettent de distinguer :
- la tendance de fond,
- les fluctuations de court terme,
- et les épisodes de volatilité accrue.

## 10) Reflection

### Difficultés rencontrées
- Gestion d’une série temporelle très longue.
- Bruit important dans les données financières.
- Non-normalité fréquente des rendements.
- Nécessité de choisir des paramètres adaptés pour les fenêtres de moyennes mobiles ou le lissage.

### Solutions apportées
- Conversion de la date en index temporel.
- Utilisation de moyennes mobiles pour réduire le bruit.
- Recours à des tests statistiques robustes.
- Application de méthodes avancées de NumPy et SciPy pour mieux interpréter les variations du marché.

### Ce que j’ai appris
- Lire et préparer une série financière.
- Visualiser les tendances et le volume.
- Construire et interpréter des tests d’hypothèse.
- Utiliser des fonctions statistiques avancées pour l’analyse des marchés financiers.

## 11) Conclusion

Ce notebook constitue une base solide pour une analyse statistique avancée d’un titre boursier.  
Vous pouvez l’améliorer encore en ajoutant :
- des bandes de Bollinger,
- une Value at Risk (VaR),
- une analyse mensuelle des rendements,
- une détection d’anomalies,
- ou une comparaison avec d’autres actions du secteur technologique.